# Prepare randomized-plate OD600 input files

This notebook pairs every `*_map.csv` plate map with the corresponding raw `*.csv` OD file in `./raw_data`, maps sample IDs to genes using `sample_key.tsv`, labels biological/technical replicates by well order, performs plate-level blank correction, and writes per-plate plus combined files to `./input`.

The original OD measurements are never overwritten. Output includes both `OD600_raw` and `OD600_blank_corrected`; the `OD600` column is selected with `APPLY_BLANK_CORRECTION` below.

## Cell 1 — Configuration

In [1]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
from IPython.display import display

# An environment variable makes the same notebook easy to test elsewhere.
WD = Path(os.environ.get(
    "BBQTL_WD",
    "/blue/juannanzhou/arindam.sarkar/BB-QTL_HT/7_HIPHOP_validation/Single-dose_QTL_randomized",
))
RAW_DIR = WD / "raw_data"
INPUT_DIR = WD / "input"

# Variable requested for the sample-ID-to-gene key.
SAMPLE_KEY_FILE = RAW_DIR / "sample_key.tsv"

# True: OD600 is blank-corrected; False: OD600 equals the original reading.
# Both raw and corrected values are retained in every long-format output.
APPLY_BLANK_CORRECTION = True

# Accepted plate-map labels for blank wells (case-insensitive).
BLANK_LABELS = {"blank", "blk", "empty"}

# Also create one-row-per-gene files with replicate columns.
WRITE_WIDE_FILES = False

if not RAW_DIR.is_dir():
    raise FileNotFoundError(f"Raw-data directory does not exist: {RAW_DIR}")
if not SAMPLE_KEY_FILE.is_file():
    raise FileNotFoundError(f"Sample key does not exist: {SAMPLE_KEY_FILE}")

INPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Working directory: {WD}")
print(f"Raw data:         {RAW_DIR}")
print(f"Output:           {INPUT_DIR}")

Working directory: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/7_HIPHOP_validation/Single-dose_QTL_randomized
Raw data:         /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/7_HIPHOP_validation/Single-dose_QTL_randomized/raw_data
Output:           /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/7_HIPHOP_validation/Single-dose_QTL_randomized/input


## Cell 2 — Load the sample key

`sample_key.tsv` may be headerless, as in the supplied example, or may begin with a `sample_id`/`gene` header row.

In [2]:
def load_sample_key(path):
    key = pd.read_csv(
        path,
        sep="\t",
        header=None,
        names=["sample_id", "gene"],
        dtype=str,
        comment="#",
    ).dropna(how="all")

    key["sample_id"] = key["sample_id"].str.strip()
    key["gene"] = key["gene"].str.strip()

    # Remove a header row if one was supplied.
    header_like = (
        key["sample_id"].str.lower().isin({"sample", "sample_id", "strain", "strain_id"})
        & key["gene"].str.lower().isin({"gene", "gene_id", "gene_key"})
    )
    key = key.loc[~header_like].copy()

    if key.empty:
        raise ValueError(f"No sample mappings were found in {path}")
    if key[["sample_id", "gene"]].isna().any().any() or (key == "").any().any():
        raise ValueError("Every sample-key row must contain both a sample_id and a gene.")
    if key["sample_id"].duplicated().any():
        duplicated = key.loc[key["sample_id"].duplicated(False), "sample_id"].unique().tolist()
        raise ValueError(f"Duplicate sample IDs in sample_key.tsv: {duplicated}")

    return key.reset_index(drop=True)


sample_key = load_sample_key(SAMPLE_KEY_FILE)
sample_to_gene = sample_key.set_index("sample_id")["gene"].to_dict()

display(sample_key)
print(f"Loaded {len(sample_key)} sample-to-gene mappings.")

,sample_id,gene
0,1,TIF11
1,2,TRM732
2,3,YMR262W
3,4,BUL1
4,5,PPA2
5,6,RRN9
6,7,SCS7
7,8,ZDS1
8,9,URA10
9,10,RSN1


Loaded 10 sample-to-gene mappings.


## Cell 3 — Discover plate-map/raw-OD pairs

For example, `CHR8_C_0x_map.csv` is paired with `CHR8_C_0x.csv`. This lets the same notebook process any number of plates.

In [3]:
map_files = sorted(RAW_DIR.glob("*_map.csv"))
if not map_files:
    raise FileNotFoundError(f"No *_map.csv files found in {RAW_DIR}")

plate_pairs = []
missing_raw_files = []
for map_file in map_files:
    raw_file = map_file.with_name(map_file.name.removesuffix("_map.csv") + ".csv")
    if raw_file.is_file():
        plate_pairs.append((map_file, raw_file))
    else:
        missing_raw_files.append(raw_file.name)

if missing_raw_files:
    raise FileNotFoundError(
        "Plate maps without matching raw OD files: " + ", ".join(missing_raw_files)
    )

pairs_table = pd.DataFrame(
    [{"plate": raw.stem, "plate_map": m.name, "raw_OD": raw.name} for m, raw in plate_pairs]
)
display(pairs_table)

,plate,plate_map,raw_OD
0,CHR13_C_0x,CHR13_C_0x_map.csv,CHR13_C_0x.csv
1,CHR13_T_1x,CHR13_T_1x_map.csv,CHR13_T_1x.csv


## Cell 4 — Functions to validate, reshape, map, and blank-correct a plate

In [4]:
def read_plate_matrix(path, *, numeric=False):
    plate = pd.read_csv(path, index_col=0, dtype=None if numeric else str)
    plate.index = plate.index.astype(str).str.strip().str.upper()
    plate.columns = plate.columns.astype(str).str.strip()

    if plate.index.has_duplicates or plate.columns.has_duplicates:
        raise ValueError(f"Duplicate row or column labels in {path.name}")

    if numeric:
        plate = plate.apply(pd.to_numeric, errors="coerce")
    else:
        plate = plate.apply(lambda col: col.str.strip())
    return plate


def plate_to_long(plate, value_name):
    # Explicit row-major reshaping keeps the well order A1, A2, ..., H12
    # and works across old and new pandas versions without stack warnings.
    long = pd.DataFrame({
        "row": np.repeat(plate.index.to_numpy(), len(plate.columns)),
        "column": np.tile(plate.columns.to_numpy(), len(plate.index)),
        value_name: plate.to_numpy().reshape(-1),
    })
    long["well"] = long["row"] + long["column"]
    return long


def process_plate(map_file, raw_file, sample_key_df):
    plate_id = raw_file.stem
    plate_map = read_plate_matrix(map_file, numeric=False)
    raw_od = read_plate_matrix(raw_file, numeric=True)

    if not plate_map.index.equals(raw_od.index) or not plate_map.columns.equals(raw_od.columns):
        raise ValueError(
            f"Row/column labels differ between {map_file.name} and {raw_file.name}"
        )

    map_long = plate_to_long(plate_map, "sample_id")
    od_long = plate_to_long(raw_od, "OD600_raw")
    plate = map_long.merge(
        od_long[["row", "column", "OD600_raw"]],
        on=["row", "column"],
        how="left",
        validate="one_to_one",
    )

    plate["sample_id"] = plate["sample_id"].astype("string").str.strip()
    if plate["sample_id"].isna().any() or (plate["sample_id"] == "").any():
        bad_wells = plate.loc[
            plate["sample_id"].isna() | (plate["sample_id"] == ""), "well"
        ].tolist()
        raise ValueError(f"Unlabeled wells in {map_file.name}: {bad_wells}")
    if plate["OD600_raw"].isna().any():
        bad_wells = plate.loc[plate["OD600_raw"].isna(), "well"].tolist()
        raise ValueError(f"Missing or non-numeric OD values in {raw_file.name}: {bad_wells}")

    plate["is_blank"] = plate["sample_id"].str.lower().isin(BLANK_LABELS)
    blank_wells = plate.loc[plate["is_blank"]].copy()
    if blank_wells.empty:
        raise ValueError(f"No blank wells were identified in {map_file.name}")

    blank_mean = blank_wells["OD600_raw"].mean()
    blank_median = blank_wells["OD600_raw"].median()
    blank_sd = blank_wells["OD600_raw"].std(ddof=1)

    measurements = plate.loc[~plate["is_blank"]].copy()
    known_ids = set(sample_key_df["sample_id"])
    observed_ids = set(measurements["sample_id"])
    unknown_ids = sorted(observed_ids - known_ids)
    if unknown_ids:
        raise ValueError(
            f"Sample IDs in {map_file.name} are absent from sample_key.tsv: {unknown_ids}"
        )

    measurements = measurements.merge(
        sample_key_df,
        on="sample_id",
        how="left",
        validate="many_to_one",
    )
    # The long table is already in row-major well order (A1, A2, ..., H12).
    measurements["replicate_number"] = (
        measurements.groupby("sample_id", sort=False).cumcount() + 1
    )
    max_replicate = int(measurements["replicate_number"].max())
    replicate_width = max(2, len(str(max_replicate)))
    measurements["replicate"] = measurements["replicate_number"].map(
        lambda n: f"replicate_{n:0{replicate_width}d}"
    )

    measurements["plate"] = plate_id
    measurements["blank_mean"] = blank_mean
    measurements["OD600_blank_corrected"] = measurements["OD600_raw"] - blank_mean
    measurements["OD600"] = (
        measurements["OD600_blank_corrected"]
        if APPLY_BLANK_CORRECTION
        else measurements["OD600_raw"]
    )

    output_columns = [
        "plate",
        "sample_id",
        "gene",
        "replicate",
        "replicate_number",
        "well",
        "OD600",
        "OD600_raw",
        "blank_mean",
        "OD600_blank_corrected",
    ]
    measurements = measurements[output_columns].sort_values(
        ["sample_id", "replicate_number"], kind="stable"
    ).reset_index(drop=True)

    blank_wells.insert(0, "plate", plate_id)
    blank_wells = blank_wells[["plate", "well", "OD600_raw"]].reset_index(drop=True)
    blank_summary = pd.DataFrame([{
        "plate": plate_id,
        "blank_n": len(blank_wells),
        "blank_mean": blank_mean,
        "blank_median": blank_median,
        "blank_sd": blank_sd,
        "blank_min": blank_wells["OD600_raw"].min(),
        "blank_max": blank_wells["OD600_raw"].max(),
    }])

    return measurements, blank_wells, blank_summary


def make_wide(long_df):
    wide = (
        long_df.pivot(
            index=["plate", "sample_id", "gene"],
            columns="replicate",
            values="OD600",
        )
        .reset_index()
        .rename_axis(columns=None)
    )
    return wide

## Cell 5 — Process every plate and write input files

Outputs are tab-separated (`.tsv`) to preserve the sample key convention and avoid ambiguity. The long files keep well IDs and both raw/corrected readings; wide files put replicate OD600 values next to each strain ID and gene.

In [5]:
all_long = []
all_blank_wells = []
all_blank_summaries = []
written_files = []

for map_file, raw_file in plate_pairs:
    long_df, blank_wells_df, blank_summary_df = process_plate(
        map_file, raw_file, sample_key
    )
    plate_id = raw_file.stem

    long_path = INPUT_DIR / f"{plate_id}_input_long.tsv"
    long_df.to_csv(long_path, sep="\t", index=False, float_format="%.6g")
    written_files.append(long_path)

    if WRITE_WIDE_FILES:
        wide_path = INPUT_DIR / f"{plate_id}_input_wide.tsv"
        make_wide(long_df).to_csv(wide_path, sep="\t", index=False, float_format="%.6g")
        written_files.append(wide_path)

    all_long.append(long_df)
    all_blank_wells.append(blank_wells_df)
    all_blank_summaries.append(blank_summary_df)

combined_long = pd.concat(all_long, ignore_index=True)
combined_blank_wells = pd.concat(all_blank_wells, ignore_index=True)
blank_summary = pd.concat(all_blank_summaries, ignore_index=True)

combined_long_path = INPUT_DIR / "all_plates_input_long.tsv"
combined_long.to_csv(combined_long_path, sep="\t", index=False, float_format="%.6g")
written_files.append(combined_long_path)

if WRITE_WIDE_FILES:
    combined_wide_path = INPUT_DIR / "all_plates_input_wide.tsv"
    make_wide(combined_long).to_csv(
        combined_wide_path, sep="\t", index=False, float_format="%.6g"
    )
    written_files.append(combined_wide_path)

blank_wells_path = INPUT_DIR / "all_blank_wells.tsv"
blank_summary_path = INPUT_DIR / "blank_summary.tsv"
combined_blank_wells.to_csv(
    blank_wells_path, sep="\t", index=False, float_format="%.6g"
)
blank_summary.to_csv(
    blank_summary_path, sep="\t", index=False, float_format="%.6g"
)
written_files.extend([blank_wells_path, blank_summary_path])

print(f"Processed {len(plate_pairs)} plate(s).")
print(f"Primary OD600 column is {'blank-corrected' if APPLY_BLANK_CORRECTION else 'raw'}.")
print("\nCreated files:")
for path in written_files:
    print(f"  {path}")

Processed 2 plate(s).
Primary OD600 column is blank-corrected.

Created files:
  /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/7_HIPHOP_validation/Single-dose_QTL_randomized/input/CHR13_C_0x_input_long.tsv
  /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/7_HIPHOP_validation/Single-dose_QTL_randomized/input/CHR13_T_1x_input_long.tsv
  /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/7_HIPHOP_validation/Single-dose_QTL_randomized/input/all_plates_input_long.tsv
  /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/7_HIPHOP_validation/Single-dose_QTL_randomized/input/all_blank_wells.tsv
  /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/7_HIPHOP_validation/Single-dose_QTL_randomized/input/blank_summary.tsv


## Cell 6 — Quality-control checks and previews

In [6]:
assert not combined_long.empty, "No non-blank sample measurements were produced."
assert combined_long[["sample_id", "gene", "replicate", "OD600"]].notna().all().all()
assert not combined_long.duplicated(["plate", "sample_id", "replicate"]).any()
assert np.isfinite(combined_long[["OD600", "OD600_raw", "blank_mean"]].to_numpy()).all()

replicate_counts = (
    combined_long.groupby(["plate", "sample_id", "gene"], as_index=False)
    .agg(replicates=("OD600", "size"), mean_OD600=("OD600", "mean"))
)

print("Blank summary")
display(blank_summary)
print("Replicate counts and mean OD600")
display(replicate_counts)
print("Preview of the combined long-format input")
display(combined_long.head(12))

print("All validation checks passed.")

Blank summary


,plate,blank_n,blank_mean,blank_median,blank_sd,blank_min,blank_max
0,CHR13_C_0x,6,0.433500,0.114,0.497697,0.11,1.082
1,CHR13_T_1x,6,0.214167,0.106,0.269864,0.10,0.765


Replicate counts and mean OD600


,plate,sample_id,gene,replicates,mean_OD600
0,CHR13_C_0x,1,TIF11,9,0.591500
1,CHR13_C_0x,10,RSN1,9,0.574056
2,CHR13_C_0x,2,TRM732,9,0.564056
3,CHR13_C_0x,3,YMR262W,9,0.597833
4,CHR13_C_0x,4,BUL1,9,0.610833
5,CHR13_C_0x,5,PPA2,9,0.575500
6,CHR13_C_0x,6,RRN9,9,0.595833
7,CHR13_C_0x,7,SCS7,9,0.609278
8,CHR13_C_0x,8,ZDS1,9,0.590611
9,CHR13_C_0x,9,URA10,9,0.559833


Preview of the combined long-format input


,plate,sample_id,gene,replicate,replicate_number,well,OD600,OD600_raw,blank_mean,OD600_blank_corrected
0,CHR13_C_0x,1,TIF11,replicate_01,1,A11,0.6035,1.037,0.4335,0.6035
1,CHR13_C_0x,1,TIF11,replicate_02,2,B5,0.5305,0.964,0.4335,0.5305
2,CHR13_C_0x,1,TIF11,replicate_03,3,C1,0.7155,1.149,0.4335,0.7155
3,CHR13_C_0x,1,TIF11,replicate_04,4,D8,0.6065,1.040,0.4335,0.6065
4,CHR13_C_0x,1,TIF11,replicate_05,5,E3,0.6365,1.070,0.4335,0.6365
5,CHR13_C_0x,1,TIF11,replicate_06,6,E12,0.5835,1.017,0.4335,0.5835
6,CHR13_C_0x,1,TIF11,replicate_07,7,F2,0.6345,1.068,0.4335,0.6345
7,CHR13_C_0x,1,TIF11,replicate_08,8,G6,0.3915,0.825,0.4335,0.3915
8,CHR13_C_0x,1,TIF11,replicate_09,9,H4,0.6215,1.055,0.4335,0.6215
9,CHR13_C_0x,10,RSN1,replicate_01,1,A10,0.5605,0.994,0.4335,0.5605


All validation checks passed.


### Output interpretation

- `*_input_long.tsv`: one non-blank well per row, with strain/sample ID, gene, replicate label, well, raw OD600, plate blank mean, corrected OD600, and the configurable primary `OD600` value.
- `*_input_wide.tsv`: one strain/gene per row, with `replicate_01`, `replicate_02`, … columns containing the primary `OD600` value.
- `all_plates_input_*`: combined versions across all discovered plates.
- `all_blank_wells.tsv` and `blank_summary.tsv`: blank-well audit and per-plate blank QC.

Blank correction is performed separately for each plate. Negative corrected values, if any, are retained rather than clipped, preserving the measurements for later QC.